In [19]:
import sqlite3
from datetime import datetime, timedelta
import random
import pandas as pd

In [20]:
# Define 21 firecracker products with categories and base prices
PRODUCTS = [
    {"product_id": "P101", "name": "Sparklers 10cm (Box)", "category": "Sparklers", "price": 40},
    {"product_id": "P102", "name": "Sparklers 30cm (Box)", "category": "Sparklers", "price": 90},
    {"product_id": "P103", "name": "Electric Sparklers Multicolor", "category": "Sparklers", "price": 120},
    {"product_id": "P104", "name": "Peacock Feather Sparklers", "category": "Sparklers", "price": 150},
    {"product_id": "P201", "name": "Flower Pots Small", "category": "Ground Novelties", "price": 180},
    {"product_id": "P202", "name": "Flower Pots Big", "category": "Ground Novelties", "price": 250},
    {"product_id": "P203", "name": "Flower Pots Special", "category": "Ground Novelties", "price": 350},
    {"product_id": "P301", "name": "Ground Chakkar Standard", "category": "Ground Novelties", "price": 60},
    {"product_id": "P302", "name": "Ground Chakkar Special", "category": "Ground Novelties", "price": 110},
    {"product_id": "P303", "name": "Ground Chakkar Deluxe", "category": "Ground Novelties", "price": 200},
    {"product_id": "P401", "name": "Atom Bomb (Standard)", "category": "Bombs", "price": 100},
    {"product_id": "P402", "name": "Atom Bomb Deluxe", "category": "Bombs", "price": 220},
    {"product_id": "P501", "name": "Rocket Bomb Single", "category": "Rockets", "price": 130},
    {"product_id": "P502", "name": "Two Sound Rocket", "category": "Rockets", "price": 260},
    {"product_id": "P503", "name": "Atomic Rocket", "category": "Rockets", "price": 400},
    {"product_id": "P601", "name": "Fancy Sky Shot 7-Shot", "category": "Aerial Shots", "price": 450},
    {"product_id": "P602", "name": "Fancy Sky Shot 12-Shot", "category": "Aerial Shots", "price": 750},
    {"product_id": "P603", "name": "Fancy Sky Shot 30-Shot", "category": "Aerial Shots", "price": 1800},
    {"product_id": "P604", "name": "Fancy Sky Shot 60-Shot", "category": "Aerial Shots", "price": 3500},
    {"product_id": "P701", "name": "Sparkler Cones", "category": "Novelties", "price": 300},
    {"product_id": "P702", "name": "Magic Pencil", "category": "Novelties", "price": 160}
]

In [21]:

# Diwali dates for Years 4 (2026) and 5 (2027)
DIWALI_DATES_DB = {
    4: datetime(2026, 11, 8),   # Year 4 (2026)
    5: datetime(2027, 10, 29)   # Year 5 (2027)
}


In [22]:

def generate_db_transactions(year_num, diwali_date):
    records = []
    start_date = diwali_date - timedelta(days=45)
    end_date = diwali_date + timedelta(days=15)
    
    tx_id_counter = 400000 * (year_num - 3) # Starts at TX400001 for Yr 4, TX500001 for Yr 5
    current_date = start_date
    
    while current_date <= end_date:
        days_to_diwali = (diwali_date - current_date).days
        
        if -2 <= days_to_diwali <= 3:
            daily_tx_count = random.randint(170, 230)
        elif -15 <= days_to_diwali < -2:
            daily_tx_count = random.randint(90, 140)
        else:
            daily_tx_count = random.randint(30, 60)
            
        for _ in range(daily_tx_count):
            tx_id_counter += 1
            prod = random.choice(PRODUCTS)
            quantity_sold = random.randint(1, 6)
            total_amount = quantity_sold * prod["price"]
            
            records.append({
                "Transaction_ID": f"TX{tx_id_counter}",
                "Date": current_date.strftime("%Y-%m-%d"),
                "Days_To_Diwali": days_to_diwali,
                "Product_ID": prod["product_id"],
                "Product_Name": prod["name"],
                "Category": prod["category"],
                "Unit_Price": prod["price"],
                "Quantity_Sold": quantity_sold,
                "Total_Revenue": total_amount,
                "Data_Source": f"Database_Year_{year_num}"
            })
            
        current_date += timedelta(days=1)
        
    df = pd.DataFrame(records)
    # Ensure exact size (~4,000 records)
    if len(df) > 4000:
        df = df.sample(n=4000, random_state=42).sort_values("Date").reset_index(drop=True)
    elif len(df) < 3000:
        extra = df.sample(n=3000 - len(df), replace=True, random_state=42)
        df = pd.concat([df, extra], ignore_index=True).sort_values("Date").reset_index(drop=True)
        
    return df


In [23]:

# Connect to SQLite database (creates fataka_sales.db in your directory)
conn = sqlite3.connect("../Dataset/fataka_sales.db")
cursor = conn.cursor()

for yr, d_date in DIWALI_DATES_DB.items():
    df_yr = generate_db_transactions(yr, d_date)
    table_name = f"transactions_year_{yr}"
    
    # Write table to SQLite database
    df_yr.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Created table '{table_name}' with {len(df_yr):,} records in SQLite database.")

conn.close()
print("All digital database records for Years 4 and 5 successfully created!")

Created table 'transactions_year_4' with 4,000 records in SQLite database.
Created table 'transactions_year_5' with 4,000 records in SQLite database.
All digital database records for Years 4 and 5 successfully created!
